# Chemistry Convolutional Neural Networks Example
In this example we will build a CNN to predict compound class from MS/MS spectra. MS/MS spectra contains m/z values for the fragments of a molecule that occur during tandem-MS/MS. Some classes of molecule, such as pepties, will have signature fragmentation patterns. Data is sourced from MassBank.

# Install Packages
Only run if you do not have packages installed

In [ ]:
%pip install rdkit
%pip install pandas
%pip install scikit-learn
%pip install matplotlib
%pip install torch torchvision torchaudio #note this will install pytorch for CPU
%pip install optuna

# Import Libraries and Set Hyperparameters
This code imports libraries, sets hyperparameters for data processing and training, sets filenames, and random seed

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import re
import random
from typing import List, Tuple

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder


#set files and hyperparameters to be used later
SPECTRA_FILE = "massbank_spectra.csv"   
CLASS_CSV = "massbank_classes.csv"         

TARGET_COLUMN = "superclass"                
TOP_N_CLASSES = 10
MIN_NUM_PEAKS = 3
MIN_CLASS_SIZE = 20

MZ_MAX = 1000.0
BIN_SIZE = 1.0

BATCH_SIZE = 64
EPOCHS = 15
LR = 1e-3
RANDOM_SEED = 42

#setting a device allows us to specify to run on cpu or gpu or to let it switch depending on if a gpu is available
#this will run on gpu if available
#because the hyperparameter optimization will be slow, running on gpu will be better
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


#set random seed for all python libraries for preproucibility
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(RANDOM_SEED)




Using device: cpu


# Read and process input files
Reads the input data, put in dataframe

In [4]:
#Parse preprocessed spectra

#this is a regular expression that allows us to easily parse our input data which is in the 
#format ('mz_1', intensity_1)
peak_pattern = re.compile(
    r"\(\s*['\"]?([0-9eE+\-\.]+)['\"]?\s*,\s*['\"]?([0-9eE+\-\.]+)['\"]?\s*\)"
)


def parse_spectrum_line(line: str):
    """
    Parse a line of the format:
    smiles,(mz1,int1),(mz2,int2),...

    Returns:
        smiles: str
        peaks: list of (mz, intensity)
    """
    line = line.strip()
    if not line:
        return None

    first_comma = line.find(",")
    if first_comma == -1:
        return None

    smiles = line[:first_comma].strip()
    peak_text = line[first_comma + 1:]

    peaks = []
    for mz_str, int_str in peak_pattern.findall(peak_text):
        try:
            mz = float(mz_str)
            intensity = float(int_str)
            peaks.append((mz, intensity))
        except ValueError:
            continue

    if not smiles or len(peaks) == 0:
        return None

    return smiles, peaks


def load_preprocessed_spectra(path: str) -> pd.DataFrame:
    rows = []

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for i, line in enumerate(f, start=1):
            parsed = parse_spectrum_line(line)
            if parsed is None:
                continue

            smiles, peaks = parsed

            if len(peaks) < MIN_NUM_PEAKS:
                continue

            rows.append((smiles, peaks, len(peaks)))


    spectra_df = pd.DataFrame(rows, columns=["smiles", "peaks", "num_peaks"])
    return spectra_df


# =========================
# Spectrum preprocessing
# =========================

def spectrum_to_binned_vector(
    peaks: List[Tuple[float, float]],
    mz_max: float = 1000.0,
    bin_size: float = 1.0,
) -> np.ndarray:
    """
    Convert a list of (mz, normalized_intensity) peaks into a fixed-length vector.
    Assumes intensities are already normalized.
    """
    n_bins = int(mz_max / bin_size)
    vec = np.zeros(n_bins, dtype=np.float32)

    if not peaks:
        return vec

    for mz, intensity in peaks:
        if mz < 0 or mz >= mz_max:
            continue
        idx = int(mz // bin_size)
        if 0 <= idx < n_bins:
            vec[idx] = max(vec[idx], float(intensity))

    # optional renormalization in case the input is not exactly scaled
    max_val = vec.max()
    if max_val > 0:
        vec /= max_val
    return vec


# =========================
# Dataset assembly
# =========================

def canonicalize_smiles_column(series: pd.Series) -> pd.Series:
    """
    Minimal cleanup only.
    If you want true canonicalization, use RDKit.
    """
    return series.astype(str).str.strip()


def build_dataset(
    spectra_file: str,
    class_csv: str,
    target_column: str = "superclass",
    top_n_classes: int = 10,
) -> pd.DataFrame:
    spectra_df = load_preprocessed_spectra(spectra_file)
    print(f"Loaded spectra: {len(spectra_df)}")

    label_df = pd.read_csv(class_csv)
    if "smiles" not in label_df.columns:
        raise ValueError("classyfire_labels.csv must contain a 'smiles' column")
    if target_column not in label_df.columns:
        raise ValueError(f"classyfire_labels.csv must contain '{target_column}'")

    spectra_df["smiles"] = canonicalize_smiles_column(spectra_df["smiles"])
    label_df["smiles"] = canonicalize_smiles_column(label_df["smiles"])

    merged = spectra_df.merge(
        label_df[["smiles", target_column]].drop_duplicates(),
        on="smiles",
        how="inner"
    )

    merged = merged.dropna(subset=[target_column]).copy()
    print(f"Spectra with labels: {len(merged)}")

    if top_n_classes is not None:
        top_classes = merged[target_column].value_counts().head(top_n_classes).index
        merged = merged[merged[target_column].isin(top_classes)].copy()

    merged["x"] = merged["peaks"].apply(
        lambda p: spectrum_to_binned_vector(
            p,
            mz_max=MZ_MAX,
            bin_size=BIN_SIZE,
        )
    )

    class_counts = merged[target_column].value_counts()
    keep_classes = class_counts[class_counts >= MIN_CLASS_SIZE].index
    merged = merged[merged[target_column].isin(keep_classes)].copy()

    print("Final class counts:")
    print(merged[target_column].value_counts())

    return merged.reset_index(drop=True)


# =========================
# Split by molecule identity
# =========================

def split_by_smiles(df: pd.DataFrame, label_col: str):
    """
    Split by unique SMILES so replicate spectra of the same molecule
    do not appear in both train and test.
    """
    mol_df = df[["smiles", label_col]].drop_duplicates()

    train_mols, test_mols = train_test_split(
        mol_df,
        test_size=0.2,
        random_state=RANDOM_SEED,
        stratify=mol_df[label_col]
    )

    train_set = df[df["smiles"].isin(train_mols["smiles"])].copy()
    test_set = df[df["smiles"].isin(test_mols["smiles"])].copy()

    return train_set, test_set


# =========================
# PyTorch dataset
# =========================

class SpectraDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)  # [N, 1, L]
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

df = build_dataset(
        spectra_file=SPECTRA_FILE,
        class_csv=CLASS_CSV,
        target_column=TARGET_COLUMN,
        top_n_classes=TOP_N_CLASSES,
    )
df


Loaded spectra: 5370
Spectra with labels: 5370
Final class counts:
superclass
Benzenoids                               1618
Organoheterocyclic compounds             1255
Lipids and lipid-like molecules          1019
Organic acids and derivatives             584
Phenylpropanoids and polyketides          332
Organic oxygen compounds                  190
Organic nitrogen compounds                159
Nucleosides nucleotides and analogues      96
Alkaloids and derivatives                  43
Organohalogen compounds                    31
Name: count, dtype: int64


,smiles,peaks,num_peaks,superclass,x
0,CC1CC2=C(C(=CC=C2)O)C(=O)O1,"[(133.0648, 0.22522522522522523), (151.0754, 0...",5,Organoheterocyclic compounds,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4[C@@H]5C=CO[C...,"[(243.0652, 0.07907907907907907), (283.0601, 0...",4,Phenylpropanoids and polyketides,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,COC1=C2C3=C(C(=O)CC3)C(=O)OC2=C4C(=C1)O[C@@H]5...,"[(257.0808, 0.07907907907907907), (259.0601, 0...",8,Phenylpropanoids and polyketides,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,CC(=O)C1=C([C@@H]2[C@@H]3[C@@H](CC4=C5C3=CNC5=...,"[(182.0812, 0.2132132132132132), (196.1121, 0....",3,Organoheterocyclic compounds,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,C[C@]12C[C@@H]([C@H](C=C1C3=CC(=CC(=C3C(=O)O2)...,"[(213.091, 0.04004004004004004), (229.0859, 0....",8,Organoheterocyclic compounds,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
...,...,...,...,...,...
5322,CCCC(N1[C@H](C(N[C@H](C(N[C@@H]([C@H](OC([C@H]...,"[(140.1065, 0.043043043043043044), (168.1016, ...",21,Organic acids and derivatives,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
5323,CCCC(N1[C@H](C(N[C@H](C(N[C@@H]([C@H](OC([C@H]...,"[(140.1064, 0.03903903903903904), (168.1015, 0...",21,Organic acids and derivatives,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
5324,CCCC(N1[C@H](C(N[C@H](C(N[C@@H]([C@H](OC([C@H]...,"[(168.1016, 0.1021021021021021), (320.1614, 0....",21,Organic acids and derivatives,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
5325,CCCC(N1[C@H](C(N[C@H](C(N[C@@H]([C@H](OC([C@H]...,"[(140.1065, 0.03403403403403404), (168.1017, 0...",20,Organic acids and derivatives,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


# Split data
Split the data and create pytorch dataset

In [5]:
train_df, val_df = split_by_smiles(df, TARGET_COLUMN)

print(f"Train spectra: {len(train_df)}")
print(f"Validation spectra:  {len(val_df)}")

#LabelEncoder numbers the class labels
le = LabelEncoder()
y_train = le.fit_transform(train_df[TARGET_COLUMN])
y_val = le.transform(val_df[TARGET_COLUMN])

X_train = np.stack(train_df["x"].values)
X_val = np.stack(val_df["x"].values)

train_ds = SpectraDataset(X_train, y_train)
val_ds = SpectraDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

Train spectra: 4261
Validation spectra:  1066


# Set up and train CNN
Try adjusting the numver of filters or layers in the CNN and see how it affects performance

In [ ]:

# =========================
# 1D CNN
# =========================

class MassSpecCNN(nn.Module):
    def __init__(self, input_length, n_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=9, padding=4),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=9, padding=4),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=9, padding=4),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
        )

        pooled_length = input_length // 8
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * pooled_length, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# =========================
# Training / evaluation
# =========================

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for Xb, yb in loader:
        #this ensures our tensors are set up properly to run on gpu or cpu
        Xb = Xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = model(Xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * Xb.size(0)

    return total_loss / len(loader.dataset)

#note this is a decorator, it will make torch.no_grad() apply to the whole function that follows
#this is an alternative to using "with torch.no_grad():" we had in the last example
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    y_true = []
    y_pred = []

    for Xb, yb in loader:
        Xb = Xb.to(device)
        logits = model(Xb)
        preds = torch.argmax(logits, dim=1).cpu().numpy()

        y_true.extend(yb.numpy())
        y_pred.extend(preds)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    return acc, bal_acc, y_true, y_pred
   

input_length = X_train.shape[1]
n_classes = len(le.classes_)

model = MassSpecCNN(input_length=input_length, n_classes=n_classes).to(DEVICE)

class_counts = np.bincount(y_train)
#set class weights to improve performance on rare classes
class_weights = len(y_train) / (len(class_counts) * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    train_acc, train_bal_acc, _, _ = evaluate(model, train_loader, DEVICE)
    val_acc, val_bal_acc, y_true, y_pred = evaluate(model, val_loader, DEVICE)

    print(
        f"Epoch {epoch:02d} | "
        f"loss={train_loss:.4f} | "
        f"train_acc={train_acc:.4f} | train_bal_acc={train_bal_acc:.4f} | "
        f"val_acc={val_acc:.4f} | val_bal_acc={val_bal_acc:.4f}"
    )

print("\nClasses:")
for i, cls in enumerate(le.classes_):
    print(i, cls)

print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=le.classes_))


Epoch 01 | loss=2.3740 | train_acc=0.2337 | train_bal_acc=0.3048 | val_acc=0.2223 | val_bal_acc=0.2243


# Optimizing with Optuna
We will now look at how to optimize hyperparameters with optuna

In [ ]:
import optuna
import copy
import os
import shutil


#this method lets us generate a list of channels with different growth strateiges
def generate_channels(n_layers, initial_channels, channel_growth):

    channels = []

    for i in range(n_layers):

        if channel_growth == "constant":
            n_channels = initial_channels

        elif channel_growth == "double":
            n_channels = initial_channels * (2 ** i)

        elif channel_growth == "add_32":
            n_channels = initial_channels + 32 * i

        else:
            raise ValueError(
                f"Unknown channel growth strategy: {channel_growth}"
            )

        channels.append(n_channels)

    return channels

#we will define our CNN again but this time with more hyparamters as arguments
class MassSpecCNN(nn.Module):
    def __init__(self, n_classes, input_length, channels=[32,64,128],kernel_size=9,hidden_size=256,dropout=0.3):
        super().__init__()
        
        #this will be a list of our layers
        layers = []
        in_channels = 1
        #iterate through our list of channel sizes and add one conv layer with that channel output
        for out_channels in channels:
            layers.extend([
                nn.Conv1d(
                    in_channels,
                    out_channels,
                    kernel_size=kernel_size,
                    padding=kernel_size // 2
                ),
                nn.BatchNorm1d(out_channels),
                nn.ReLU(),
                nn.MaxPool1d(2)
            ])

            in_channels = out_channels
        
        #now we turn our layer list into the features part of our network
        self.features = nn.Sequential(*layers)
        
        # We need to calculate the flattened size for our specific CNN to know what the linear input size is
        # to do this we can run dummy features through the network
        with torch.no_grad():
            dummy_input = torch.zeros(
                1, 1, input_length
            )

            # Temporarily set to eval mode for BatchNorm
            self.features.eval()
            dummy_output = self.features(dummy_input)
            self.features.train()

            flattened_size = dummy_output.flatten(1).shape[1]
            
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flattened_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, n_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
    

#define an optimization process for Optuna
def objective(trial):

    # ---------------------------------
    # Architecture hyperparameters
    # ---------------------------------
    
    #the suggest functions suggest what initial values Optuna should look at
    #suggest_int chooses integers from a range
    #suggest_categorical chooses one of the options in th elist
    
    #number of layers in the CNN
    n_layers = trial.suggest_int(
        "n_layers", 1, 3
    )

    initial_channels = trial.suggest_categorical(
        "initial_channels", [16, 32, 64]
    )

    # How channels change between layers
    # this will let us allow the same number of channels at each convolution or for them to grow following a set scheme
    channel_growth = trial.suggest_categorical(
        "channel_growth",
        ["constant", "double", "add_32"]
    )
    
    # Construct channel list
    channels = generate_channels(n_layers, initial_channels, channel_growth)

    kernel_size = trial.suggest_categorical(
        "kernel_size", [3, 5, 7]
    )

    hidden_size = trial.suggest_categorical(
        "hidden_size", [32, 64, 128, 256]
    )

    dropout = trial.suggest_float(
        "dropout", 0.0, 0.5
    )

    # ---------------------------------
    # Training hyperparameters
    # ---------------------------------
    #samples from this range on a log scale
    learning_rate = trial.suggest_float(
        "learning_rate", 1e-5, 1e-2, log=True
    )

    weight_decay = trial.suggest_float(
        "weight_decay", 1e-6, 1e-2, log=True
    )

    batch_size = trial.suggest_categorical(
        "batch_size", [32, 64, 128]
    )

    # ---------------------------------
    # Build model
    # ---------------------------------

    model = MassSpecCNN(
        n_classes=n_classes,
        input_length=input_length,
        channels=channels,
        kernel_size=kernel_size,
        hidden_size=hidden_size,
        dropout=dropout
    ).to(DEVICE)

    # ---------------------------------
    # Create optimizer
    # ---------------------------------
    #we will also use a new hyperparameter for the optimizer here
    # it is weight decay, a regularization penalty that makes weights smaller
    # it is recommended to use a variation of Adam, AdamW when using weight decay
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    criterion = nn.CrossEntropyLoss()

    # ---------------------------------
    # Training loop
    # ---------------------------------

    best_accuracy = 0.0

    for epoch in range(20):

        model.train()

        for X, y in train_loader:

            X = X.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad()

            predictions = model(X)

            loss = criterion(predictions, y)

            loss.backward()
            optimizer.step()

        # ---------------------------------
        # Validation
        # ---------------------------------

        model.eval()

        correct = 0
        total = 0

        with torch.no_grad():

            for X, y in val_loader:

                X = X.to(DEVICE)
                y = y.to(DEVICE)

                predictions = model(X)

                predicted_classes = predictions.argmax(dim=1)

                correct += (predicted_classes == y).sum().item()
                total += y.size(0)

        accuracy = correct / total

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            epoch_of_best_accuracy = epoch
            best_model_state = copy.deepcopy(model.state_dict())
            


        # Report progress for pruning
        trial.report(accuracy, epoch)

        if trial.should_prune():
            raise optuna.TrialPruned()
    

    trial.set_user_attr("best_epoch", epoch_of_best_accuracy)
    # Save the weights from the best epoch in this trial
    torch.save(
        best_model_state,
        f"trial_{trial.number}.pth"
    )
    return best_accuracy


#run Optuna
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(n_startup_trials=10,multivariate=True,seed=42),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=5
    )
)

study.optimize(objective, n_trials=25)
best_trial = study.best_trial
print("Best trial:", best_trial.number)
print("Best accuracy:", best_trial.value)
print("Best epoch:", best_trial.user_attrs["best_epoch"])


# Get more detailed metrics for best model

In [ ]:
#reconstruct the best model
best_params = study.best_params

n_layers = best_params["n_layers"]

initial_channels = best_params["initial_channels"]
channel_growth = best_params["channel_growth"]

channels = generate_channels(n_layers, initial_channels, channel_growth)

best_model = MassSpecCNN(
    n_classes=n_classes,
    input_length=input_length,
    channels=channels,
    kernel_size=best_params["kernel_size"],
    hidden_size=best_params["hidden_size"],
    dropout=best_params["dropout"]
).to(DEVICE)

# Load weights from the best trial
best_model.load_state_dict(
    torch.load(
        f"trial_{study.best_trial.number}.pth",
        map_location=DEVICE,
        weights_only=True
    )
)

#evaluate model
best_model.eval()
y_true = []
y_pred = []
correct = 0
total = 0
for Xb, yb in val_loader:
    Xb = Xb.to(DEVICE)
    logits = best_model(Xb)
    preds = torch.argmax(logits, dim=1).cpu().numpy()

    y_true.extend(yb.numpy())
    y_pred.extend(preds)

    correct += (preds == yb).sum().item()
    total += yb.size(0)

accuracy = correct / total
print("Accuracy")
print(accuracy)
y_true = np.array(y_true)
y_pred = np.array(y_pred)


print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=le.classes_))